# Revisión de los errores de descarga de créditos

En este notebook reviso las consultas de créditos que fallaron durante
la descarga completa del notebook `01_obtencion_datos`. De 681.868
películas, 681.836 tienen sus créditos guardados y 32 consultas
terminaron con error.

Primero leo el registro de incidencias para identificar la causa de
cada fallo. Después decidiré cuáles reintentar y cuáles documentar
como películas sin créditos.

## 1. Lectura del registro de errores

Leo el archivo `errores_creditos.jsonl`, que contiene una incidencia
por línea. Esta celda solo lee el archivo: no consulta la API ni
modifica ningún dato.

In [9]:
# Ponlo en True solo si quieres repetir la comprobación (tarda más de una hora)
EJECUTAR_COMPROBACION = True

if EJECUTAR_COMPROBACION:
    archivos = list(carpeta_creditos.glob("creditos_*.json"))
    print("Archivos de créditos:", len(archivos), "(esperados: 681836)")

    corruptos = []
    for r in archivos:
        try:
            with open(r, "r", encoding="utf-8") as f:
                json.load(f)
        except Exception:
            corruptos.append(r.name)

    print("Archivos ilegibles:", len(corruptos))
    print("Archivos .tmp sueltos:", len(list(carpeta_creditos.glob("*.tmp"))))

Archivos de créditos: 681836 (esperados: 681836)


KeyboardInterrupt: 

El registro contiene 32 líneas que corresponden a 32 películas
distintas. Todas las incidencias son errores 404 (Not Found): la API
no devuelve créditos para esas películas, por lo que reintentar la
consulta no cambiaría el resultado.

Estas 32 películas representan el 0,005 % del total. Se conservan en
el listado original, pero se documentan como «Sin créditos descargados»
y se excluyen del análisis de dirección. Siguen siendo válidas para
el análisis general de géneros, que no depende de los créditos.

In [8]:
# Comprobamos que todos los errores son 404
es_404 = err["error"].str.startswith("404")
print("Errores 404:", es_404.sum(), "de", len(err))

# Guardamos los ids sin créditos para documentarlos
ruta_sin_creditos = carpeta_proc / "ids_sin_creditos_404.csv"
err[["id", "error"]].drop_duplicates("id").to_csv(ruta_sin_creditos, index=False)
print("Guardado en:", ruta_sin_creditos.relative_to(raiz))

NameError: name 'err' is not defined

Se confirma que las 32 incidencias son errores 404. La lista de
identificadores se guarda en `ids_sin_creditos_404.csv` para poder
documentar estas películas en las limitaciones del análisis.

## 2. Comprobación de los archivos guardados

Antes de construir la tabla de dirección, compruebo que todos los
archivos de créditos se leen correctamente y que su número coincide
con el esperado: 681.836, es decir, las 681.868 películas menos las
32 que devolvieron un error 404.

Esta comprobación abre y lee uno por uno los 681.836 archivos, por lo
que tarda más de una hora. El resultado que aparece debajo se obtuvo
en la ejecución original y no hace falta repetirlo.

In [5]:
import json
from pathlib import Path
from dotenv import find_dotenv

raiz = Path(find_dotenv(usecwd=True)).resolve().parent
carpeta_creditos = raiz / "src" / "data" / "raw" / "creditos"

archivos = list(carpeta_creditos.glob("creditos_*.json"))
print("Archivos de créditos:", len(archivos), "(esperados: 681836)")

corruptos = []
for r in archivos:
    try:
        with open(r, "r", encoding="utf-8") as f:
            json.load(f)
    except Exception:
        corruptos.append(r.name)

print("Archivos ilegibles:", len(corruptos))
print("Archivos .tmp sueltos:", len(list(carpeta_creditos.glob("*.tmp"))))

Archivos de créditos: 681836 (esperados: 681836)


KeyboardInterrupt: 

Los 681.836 archivos de créditos se leen correctamente y no quedan
archivos temporales. Los datos de dirección están listos para
construir la tabla de directores en el notebook `02b_direccion`.